# Taller 2 — Predicción Temprana de Abandono (Churn) y Generación de Valor en Banca Minorista
**MINE-4101: Ciencia de Datos Aplicada — Universidad de los Andes**  
**Consultora Analítica:** FinAlpes S.A.S. (Práctica de Inteligencia de Negocios y Ciencia de Datos Financiera)

---

## Resumen de Contexto y Problema de Negocio

En un sistema financiero altamente maduro como el colombiano —donde el **96,5 % de la población adulta** (37,9 millones de personas) ya posee al menos un producto de depósito y el **51,6 %** (20,3 millones) accede a productos de crédito formal (Superintendencia Financiera de Colombia, 2025)— el crecimiento orgánico por adquisición de nuevos usuarios presenta costos marginales crecientes. En este entorno competitivo, **retener a un cliente actual es entre 5 y 7 veces más económico que adquirir uno nuevo**, y un cliente que abandona la entidad rara vez se recupera.

**FinAlpes** ha sido contratada para desarrollar e implementar una solución analítica de **detección temprana de abandono (*Churn*)** a partir de perfiles comerciales y demográficos. El objetivo es focalizar campañas preventivas de retención para una entidad bancaria objetivo con los siguientes parámetros de negocio:

| Parámetro Estratégico | Valor / Rango de Negocio | Implicación Analítica y Financiera |
| :--- | :--- | :--- |
| **Base objetivo del banco cliente** | $N = 20.000$ clientes activos | Escalabilidad directa del modelo desde la prueba de concepto (PoC). |
| **Tasa base de deserción en la industria** | $p_{\text{base}} \approx 15,0\%$ anual ($3.000$ clientes/año) | Desbalance de clases moderado-alto; requiere ajuste por *Prior Probability Shift* respecto a la muestra europea. |
| **Efectividad de la campaña de retención** | $\gamma = 30,0\%$ de éxito sobre desertores reales contactados ($TP$) | Solo se recupera 1 de cada 3,33 clientes en riesgo real que reciben la intervención. |
| **Estructura de costos de campaña** | Cobro del $100\%$ de los clientes contactados ($TP + FP$) | Los **Falsos Positivos ($FP$)** erosionan directamente el ROI al consumir presupuesto e incentivos en clientes que no iban a abandonar. |
| **Ingreso anual por cliente activo ($V$)** | Entre $\text{COP } \$500.000$ y $\text{COP } \$900.000$ | Rango sustentado en reportes financieros de neobancos regionales (Nu Holdings, SEC Form 6-K). |

---

### Estructura del Entregable
1. **[10%] Entendimiento y Preparación de los Datos:** Carga resiliente, auditoría de calidad (duplicados, inconsistencias tipográficas, anomalías de dominio y valores faltantes), análisis exploratorio visual e ingeniería de características sin fuga de información (*Data Leakage*).
2. **[15%] Entrenamiento del Modelo de Machine Learning:** Partición estratificada 70/30, *Pipelines* reproducibles y comparación de cuatro arquitecturas (`LogisticRegression`, `RandomForest`, `XGBoost` y `LightGBM`) con validación cruzada estratificada y optimización de hiperparámetros.
3. **[20%] Análisis Cuantitativo de Resultados:** Justificación matemática y financiera de métricas (`PR-AUC`, `F1-Score`, `Recall`, `Precision` vs. `Accuracy`), optimización del umbral de decisión ($\tau$) y selección del modelo campeón.
4. **[20%] Análisis Cualitativo e Interpretabilidad (SHAP):** Explicabilidad global (*Summary Beeswarm*, importancia absoluta y gráficos de dependencia no lineal) e individual (*Waterfall plots* sobre arquetipos reales de clientes).
5. **[20%] Generación de Valor y Punto de Equilibrio (*Break-Even*):** Modelamiento paramétrico del ROI, estructura de costos por canal de contacto en Colombia, costos de infraestructura MLOps, punto de equilibrio y limitaciones técnicas/estratégicas.
6. **[10%] Insights y Recomendación Ejecutiva:** Hoja de ruta accionable por segmento de riesgo para la Vicepresidencia Comercial del banco.
7. **[5%] Declaración de Uso de IA Generativa:** Transparencia metodológica y registro de validación.

---
## 1. [10%] Entendimiento y Preparación de los Datos

### 1.1 Configuración del Entorno y Carga Resiliente de Datos
Siguiendo estándares de ingeniería de software para ciencia de datos, implementamos una función de ingesta resiliente que prioriza el archivo local `../data/raw/churn.csv` (o `data/raw/churn.csv` según el directorio de trabajo) y dispone de un mecanismo automático de *fallback* hacia el repositorio remoto de la Universidad de los Andes.

In [ ]:
import os
import sys
import warnings
from pathlib import Path

# Configuración de caché de Matplotlib y supresión de advertencias no críticas
os.environ.setdefault("MPLCONFIGDIR", str(Path.cwd() / ".mplcache"))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    GridSearchCV,
    cross_val_predict
)
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    precision_recall_curve
)
import shap

# Semilla global para reproducibilidad estricta
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Configuración estética ejecutiva para visualizaciones
plt.style.use("seaborn-v0_8-whitegrid")
PALETTE_CORP = {
    "primary": "#1B365D",    # Azul corporativo FinAlpes
    "secondary": "#008080",  # Teal analítico
    "danger": "#D9534F",     # Rojo alerta (Churn = 1)
    "safe": "#2E8B57",       # Verde retención (Churn = 0)
    "accent": "#E69F00",     # Ámbar destacado
    "neutral": "#6C757D"     # Gris neutro
}
plt.rcParams.update({
    "figure.figsize": (11, 5.5),
    "figure.dpi": 120,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "font.size": 10
})

# Resolución de rutas del proyecto (compatible desde raíz o desde /notebooks)
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DATA_PATH = Path("../data/raw/churn.csv") if Path.cwd().name == "notebooks" else Path("data/raw/churn.csv")
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

FALLBACK_URL = "https://datascience-uniandes.github.io/PracticasMachineLearning/datos/churn.csv"


def load_churn_data(local_path: Path = RAW_DATA_PATH, fallback_url: str = FALLBACK_URL) -> pd.DataFrame:
    """
    Carga resiliente del dataset de Churn.
    Intenta primero la ruta local prioritaria (../data/raw/churn.csv) y,
    en caso de no existir o fallar, recurre a la URL remota oficial del curso.
    """
    candidate_paths = [local_path, PROJECT_ROOT / "data" / "raw" / "churn.csv"]
    for path in candidate_paths:
        if path.exists():
            print(f"[INFO] Datos cargados exitosamente desde almacenamiento local: {path}")
            return pd.read_csv(path)
    print(f"[WARN] Archivo local no encontrado. Descargando desde fallback remoto: {fallback_url}")
    df_remote = pd.read_csv(fallback_url)
    # Persistir copia local para ejecuciones futuras
    target_save = PROJECT_ROOT / "data" / "raw" / "churn.csv"
    target_save.parent.mkdir(parents=True, exist_ok=True)
    df_remote.to_csv(target_save, index=False)
    return df_remote


df_raw = load_churn_data()
print(f"Dimensiones originales del dataset bruto: {df_raw.shape[0]:,} filas x {df_raw.shape[1]} columnas")
display(df_raw.head())

### 1.2 Diccionario de Datos y Auditoría de Calidad
El conjunto de datos recopila el perfil comercial y demográfico de clientes bancarios europeos en **11 variables** (10 predictoras y 1 variable objetivo binaria `churn`):

| Variable | Tipo de Dato | Naturaleza | Descripción de Negocio y Regla de Dominio |
| :--- | :--- | :--- | :--- |
| `credit_score` | Numérica (`float64`) | Continua | Puntaje de riesgo crediticio del cliente. Rango estándar válido: $[300, 850]$. |
| `country` | Categórica (`object`) | Nominal | País de residencia del cliente (`France`, `Germany`, `Spain`). |
| `gender` | Categórica (`object`) | Nominal | Género reportado del cliente (`Male`, `Female`). |
| `age` | Numérica (`float64`) | Discreta | Edad del titular en años cumplidos. Rango bancario adulto razonable: $[18, 100]$. |
| `tenure` | Numérica (`int64`) | Discreta | Antigüedad del cliente en el banco (años). Rango observado de diseño: $[0, 10]$. |
| `balance` | Numérica (`float64`) | Continua | Saldo disponible en cuentas de depósito (EUR). Rango válido: $\ge 0$. |
| `products_number` | Numérica (`int64`) | Discreta | Número de productos financieros contratados con el banco ($1$ a $4$). |
| `credit_card` | Binaria (`int64`) | Indicadora | Tenencia de tarjeta de crédito con la entidad ($1 = \text{Sí}, 0 = \text{No}$). |
| `active_member` | Binaria (`int64`) | Indicadora | Estado de actividad transaccional/comercial reciente ($1 = \text{Activo}, 0 = \text{Inactivo}$). |
| `estimated_salary` | Numérica (`float64`) | Continua | Ingreso anual estimado del cliente (EUR). Rango válido: $> 0$. |
| `churn` | Binaria (`int64`) | **Objetivo ($y$)** | Indicador de abandono ($1 = \text{Abandonó el banco}, 0 = \text{Permanece activo}$). |

Realizamos a continuación un diagnóstico exhaustivo para detectar **registros duplicados**, **inconsistencias de texto en variables categóricas**, **valores faltantes (`NaN`)** y **anomalías de dominio (valores fuera de rango físico o financiero)**.

In [ ]:
# 1. Diagnóstico de duplicados exactos
n_duplicates = df_raw.duplicated().sum()
print(f"1. Registros duplicados exactos detectados: {n_duplicates} ({n_duplicates / len(df_raw):.2%})")

# 2. Diagnóstico de inconsistencias en variables categóricas
print("\n2. Valores únicos en variables categóricas (antes de estandarizar):")
print("   - country:", df_raw["country"].value_counts(dropna=False).to_dict())
print("   - gender :", df_raw["gender"].value_counts(dropna=False).to_dict())

# 3. Diagnóstico de valores faltantes originales
nulls_raw = df_raw.isnull().sum()
nulls_df = pd.DataFrame({
    "Faltantes (Bruto)": nulls_raw,
    "% Faltantes (Bruto)": (nulls_raw / len(df_raw) * 100).round(2)
})
print("\n3. Resumen de valores faltantes en el dataset bruto:")
display(nulls_df[nulls_df["Faltantes (Bruto)"] > 0])

# 4. Diagnóstico de anomalías de dominio en variables numéricas
domain_checks = {
    "credit_score (< 300 o > 850)": df_raw[(df_raw["credit_score"] < 300) | (df_raw["credit_score"] > 850)][["credit_score", "country", "age", "churn"]],
    "age (< 18 o > 100)": df_raw[(df_raw["age"] < 18) | (df_raw["age"] > 100)][["age", "credit_score", "tenure", "churn"]],
    "tenure (< 0 o > 10)": df_raw[(df_raw["tenure"] < 0) | (df_raw["tenure"] > 10)][["tenure", "age", "country", "churn"]],
    "estimated_salary (< 0)": df_raw[df_raw["estimated_salary"] < 0][["estimated_salary", "balance", "country", "churn"]]
}

print("\n4. Auditoría de anomalías de dominio (valores imposibles o corruptos):")
for rule_name, subset in domain_checks.items():
    print(f"   * {rule_name}: {len(subset)} registros")
    display(subset)

### 1.3 Saneamiento Estructural sin Fuga de Información (*Zero Data Leakage*)
La auditoría de calidad revela cuatro hallazgos críticos que distorsionarían el entrenamiento si no se corrigen adecuadamente:
1. **45 filas duplicadas exactas:** Al eliminarlas, la muestra pasa de $10.045$ a exactamente **$10.000$ clientes únicos**.
2. **Ruido tipográfico en categorías:** `country` presenta variantes en mayúsculas y con espacios al final (`'FRANCE'`, `'france '`, `'GERMANY'`, `'germany '`, `'SPAIN'`, `'spain '`), mientras que `gender` mezcla `'Male'`/`'Female'` con `'M'`/`'F'`. Si se aplicara *One-Hot Encoding* directamente, se crearían columnas espurias y dispersas.
3. **16 anomalías severas de dominio:**
   - **Puntajes crediticios imposibles:** $95, 120, 999$ y $1200$ (4 casos fuera del estándar $[300, 850]$).
   - **Edades biológicas o contractuales inválidas:** $-1, 0, 5, 130, 150$ y $210$ años (6 casos).
   - **Antigüedades corruptas:** $-2, 15$ y $25$ años (3 casos; por ejemplo, el registro `5775` corresponde a una cliente de 20 años de edad con 25 años de antigüedad en el banco).
   - **Salarios estimados negativos:** $-5.000, -120.5$ y $-1.0$ EUR (3 casos).
4. **Valores faltantes (`NaN`):** Existen faltantes entre el $2,0\%$ y el $3,5\%$ en `balance` ($200$), `credit_score` ($250$), `age` ($300$) y `estimated_salary` ($350$).

> **Decisión Metodológica (Prevención de *Data Leakage*):**  
> En lugar de eliminar las filas con valores faltantes o anomalías (lo que destruiría más del $10\%$ de la base de clientes e impediría predecir sobre registros incompletos en producción), convertimos las anomalías de dominio en `np.nan` y **delegamos toda la imputación estadística (por mediana) exclusivamente al `Pipeline` de Scikit-Learn después de realizar la partición 70/30**. De este modo, las medianas de imputación y los parámetros de escalamiento se aprenden únicamente sobre `X_train` ($7.000$ clientes).

In [ ]:
def clean_structural_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Realiza la limpieza determinista (sin estadísticas poblacionales para evitar Data Leakage):
    1. Elimina duplicados exactos.
    2. Estandariza cadenas de texto en 'country' y 'gender'.
    3. Enmascara como NaN los valores numéricos que violan reglas de dominio bancario.
    """
    df_clean = df.drop_duplicates().copy().reset_index(drop=True)
    
    # Estandarización de variables categóricas
    df_clean["country"] = df_clean["country"].astype(str).str.strip().str.title()
    df_clean["gender"] = (
        df_clean["gender"]
        .astype(str)
        .str.strip()
        .replace({"M": "Male", "F": "Female", "m": "Male", "f": "Female"})
    )
    
    # Enmascaramiento de anomalías de dominio a NaN (se imputarán dentro del Pipeline con datos de Train)
    df_clean.loc[(df_clean["credit_score"] < 300) | (df_clean["credit_score"] > 850), "credit_score"] = np.nan
    df_clean.loc[(df_clean["age"] < 18) | (df_clean["age"] > 100), "age"] = np.nan
    df_clean.loc[(df_clean["tenure"] < 0) | (df_clean["tenure"] > 10), "tenure"] = np.nan
    df_clean.loc[df_clean["estimated_salary"] < 0, "estimated_salary"] = np.nan
    df_clean.loc[df_clean["balance"] < 0, "balance"] = np.nan
    
    return df_clean


df_clean = clean_structural_data(df_raw)

# Guardar versión estructuralmente limpia en data/processed
df_clean.to_csv(PROCESSED_DATA_DIR / "churn_cleaned.csv", index=False)

print(f"Dimensiones tras limpieza estructural: {df_clean.shape[0]:,} clientes únicos x {df_clean.shape[1]} columnas")
print(f"Tasa global de Churn en la muestra limpia: {df_clean['churn'].mean():.2%} ({df_clean['churn'].sum():,} abandonos de {len(df_clean):,})")
display(df_clean.describe().T.round(2))

### 1.4 Análisis Exploratorio de Datos (EDA) e Indicadores Clave
Exploramos visualmente el comportamiento de las variables frente al evento de deserción (`churn`) para descubrir patrones tempranos que guíen tanto la ingeniería de características como el diseño de campañas de FinAlpes.

In [ ]:
# Gráfico 1: Calidad de datos y distribución de la variable objetivo
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

# Panel A: Porcentaje de nulos (originales + anomalías enmascaradas)
null_pct = (df_clean.isnull().sum() / len(df_clean) * 100).sort_values(ascending=True)
null_pct_nonzero = null_pct[null_pct > 0]
bars = axes[0].barh(
    null_pct_nonzero.index,
    null_pct_nonzero.values,
    color=PALETTE_CORP["primary"],
    edgecolor="black",
    alpha=0.88
)
axes[0].set_title("A. Variables con Datos Faltantes / Anomalías Enmascaradas (%)")
axes[0].set_xlabel("Porcentaje de Registros Afectados (%)")
axes[0].set_xlim(0, max(null_pct_nonzero.values) * 1.22)
for bar in bars:
    w = bar.get_width()
    axes[0].text(w + 0.08, bar.get_y() + bar.get_height()/2, f"{w:.2f}% ({int(round(w*100))} reg.)", va="center", fontsize=9.5, fontweight="bold")

# Panel B: Balance de clases (Muestra Europea vs. Benchmark Colombia)
churn_counts = df_clean["churn"].value_counts().sort_index()
churn_pcts = churn_counts / len(df_clean) * 100
labels = ["Retenidos (Churn = 0)", "Desertores (Churn = 1)"]
colors = [PALETTE_CORP["safe"], PALETTE_CORP["danger"]]
bars2 = axes[1].bar(labels, churn_pcts.values, color=colors, edgecolor="black", width=0.52, alpha=0.9)
axes[1].axhline(15.0, color=PALETTE_CORP["accent"], linestyle="--", linewidth=2, label="Benchmark Industria Colombia (15,0%)")
axes[1].set_title("B. Distribución de la Variable Objetivo (Churn)")
axes[1].set_ylabel("Porcentaje de Clientes (%)")
axes[1].set_ylim(0, 95)
axes[1].legend(loc="upper right", frameon=True)
for bar, count in zip(bars2, churn_counts.values):
    h = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2, h + 2.0, f"{h:.2f}%\n({count:,} clientes)", ha="center", fontsize=10, fontweight="bold")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "01_calidad_datos_nulos.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Gráfico 2: Tasas de deserción por segmentos estratégicos y distribuciones numéricas clave
fig, axes = plt.subplots(2, 3, figsize=(16, 9.5))

# 1. Churn por País
country_stats = df_clean.groupby("country")["churn"].agg(["count", "mean"]).reset_index()
country_stats["mean_pct"] = country_stats["mean"] * 100
sns.barplot(data=country_stats, x="country", y="mean_pct", ax=axes[0, 0], palette=[PALETTE_CORP["primary"], PALETTE_CORP["danger"], PALETTE_CORP["secondary"]], edgecolor="black")
axes[0, 0].axhline(20.37, color="gray", linestyle="--", label="Promedio Muestra (20,4%)")
axes[0, 0].set_title("1. Tasa de Churn por País (%)")
axes[0, 0].set_xlabel("País")
axes[0, 0].set_ylabel("Tasa de Churn (%)")
axes[0, 0].set_ylim(0, 40)
for idx, row in country_stats.iterrows():
    axes[0, 0].text(idx, row["mean_pct"] + 1.2, f"{row['mean_pct']:.1f}%\n(n={row['count']:,})", ha="center", fontsize=9, fontweight="bold")

# 2. Churn por Número de Productos
prod_stats = df_clean.groupby("products_number")["churn"].agg(["count", "mean"]).reset_index()
prod_stats["mean_pct"] = prod_stats["mean"] * 100
bar_colors = [PALETTE_CORP["accent"], PALETTE_CORP["safe"], PALETTE_CORP["danger"], "#8B0000"]
sns.barplot(data=prod_stats, x="products_number", y="mean_pct", ax=axes[0, 1], palette=bar_colors, edgecolor="black")
axes[0, 1].set_title("2. Tasa de Churn según Número de Productos")
axes[0, 1].set_xlabel("Número de Productos Contratados")
axes[0, 1].set_ylabel("Tasa de Churn (%)")
axes[0, 1].set_ylim(0, 115)
for idx, row in prod_stats.iterrows():
    axes[0, 1].text(idx, row["mean_pct"] + 2.5, f"{row['mean_pct']:.1f}%\n(n={row['count']:,})", ha="center", fontsize=9, fontweight="bold")

# 3. Churn por Miembro Activo y Género
act_gen = df_clean.groupby(["active_member", "gender"])["churn"].mean().unstack() * 100
act_gen.index = ["Inactivo (0)", "Activo (1)"]
act_gen.plot(kind="bar", ax=axes[0, 2], color=[PALETTE_CORP["danger"], PALETTE_CORP["primary"]], edgecolor="black", width=0.65, rot=0)
axes[0, 2].set_title("3. Tasa de Churn por Actividad y Género")
axes[0, 2].set_xlabel("Estado de Actividad Comercial")
axes[0, 2].set_ylabel("Tasa de Churn (%)")
axes[0, 2].set_ylim(0, 38)
axes[0, 2].legend(title="Género")
for container in axes[0, 2].containers:
    axes[0, 2].bar_label(container, fmt="%.1f%%", padding=3, fontsize=9, fontweight="bold")

# 4. Distribución de Edad por Churn
sns.kdeplot(data=df_clean, x="age", hue="churn", common_norm=False, fill=True, palette=[PALETTE_CORP["safe"], PALETTE_CORP["danger"]], alpha=0.4, ax=axes[1, 0], linewidth=2)
axes[1, 0].set_title("4. Densidad de Edad por Estado de Churn")
axes[1, 0].set_xlabel("Edad del Cliente (Años)")
axes[1, 0].set_ylabel("Densidad")
axes[1, 0].legend(title="Churn", labels=["Abandona (1)", "Permanece (0)"])

# 5. Distribución de Saldo (Balance) por Churn
sns.boxplot(data=df_clean, x="churn", y="balance", palette=[PALETTE_CORP["safe"], PALETTE_CORP["danger"]], ax=axes[1, 1], width=0.45)
axes[1, 1].set_xticklabels(["Permanece (0)", "Abandona (1)"])
axes[1, 1].set_title("5. Distribución de Saldo en Cuenta (Balance)")
axes[1, 1].set_xlabel("Estado del Cliente")
axes[1, 1].set_ylabel("Saldo en Depósito (EUR)")

# 6. Distribución de Puntaje de Crédito por Churn
sns.violinplot(data=df_clean, x="churn", y="credit_score", palette=[PALETTE_CORP["safe"], PALETTE_CORP["danger"]], inner="quartile", ax=axes[1, 2])
axes[1, 2].set_xticklabels(["Permanece (0)", "Abandona (1)"])
axes[1, 2].set_title("6. Distribución de Credit Score por Churn")
axes[1, 2].set_xlabel("Estado del Cliente")
axes[1, 2].set_ylabel("Puntaje de Crédito")

plt.suptitle("Análisis Exploratorio Multidimensional: Perfilamiento del Abandono Bancario", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "02_eda_distribuciones_churn.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Gráfico 3: Matriz de correlación e interacciones clave (Edad vs. Productos y País vs. Balance)
fig, axes = plt.subplots(1, 2, figsize=(14, 5.2))

# Codificación temporal solo para visualización de correlaciones
df_corr = df_clean.copy()
df_corr["is_germany"] = (df_corr["country"] == "Germany").astype(int)
df_corr["is_female"] = (df_corr["gender"] == "Female").astype(int)
corr_cols = ["credit_score", "age", "tenure", "balance", "products_number", "credit_card", "active_member", "estimated_salary", "is_germany", "is_female", "churn"]
corr_matrix = df_corr[corr_cols].corr()

sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-0.4, vmax=0.4, ax=axes[0], cbar_kws={"shrink": 0.8})
axes[0].set_title("A. Matriz de Correlación Lineal de Pearson")

# Tasa de Churn por tramo de Edad y Actividad
df_corr["age_band"] = pd.cut(df_corr["age"], bins=[17, 30, 40, 50, 60, 100], labels=["18-30", "31-40", "41-50", "51-60", "60+"])
pivot_age_act = df_corr.pivot_table(index="age_band", columns="active_member", values="churn", aggfunc="mean") * 100
pivot_age_act.columns = ["Inactivo (0)", "Activo (1)"]
pivot_age_act.plot(kind="bar", ax=axes[1], color=[PALETTE_CORP["danger"], PALETTE_CORP["safe"]], edgecolor="black", rot=0, width=0.68)
axes[1].set_title("B. Interacción No Lineal: Tramo de Edad vs. Estado de Actividad")
axes[1].set_xlabel("Rango de Edad (Años)")
axes[1].set_ylabel("Tasa de Churn (%)")
axes[1].set_ylim(0, 75)
for container in axes[1].containers:
    axes[1].bar_label(container, fmt="%.1f%%", padding=3, fontsize=9, fontweight="bold")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "03_eda_correlacion_multivariada.png", dpi=150, bbox_inches="tight")
plt.show()

### 1.5 Hallazgos Ejecutivos del Entendimiento de Datos
1. **La Paradoja de la Venta Cruzada (`products_number`):** Los clientes con **2 productos** presentan la menor tasa de deserción de todo el banco (**7,58 %**), consolidándose como el *punto óptimo de vinculación*. Por el contrario, los clientes monoproducto (`1 producto`) desertan al **27,71 %**, y aquellos con **3 o 4 productos** sufren tasas de abandono críticas del **82,71 %** y **100,0 %** respectivamente. Esto evidencia una relación fuertemente **no lineal (en forma de U invertida/J)** que un modelo lineal simple sin ingeniería de variables no podría capturar.
2. **Vulnerabilidad Demográfica por Edad (`age`) e Inactividad (`active_member`):** La distribución de edades de los desertores se desplaza marcadamente hacia el rango de **41 a 60 años**. En particular, entre los clientes de **51 a 60 años inactivos**, la tasa de abandono supera el **65 %**, mientras que en menores de 30 años es inferior al **8 %**.
3. **Brecha Regional (`country` = Germany) y de Saldo (`balance`):** Alemania registra una tasa de abandono del **32,44 %**, duplicando a Francia (**16,15 %**) y España (**16,67 %**). Además, los clientes con saldo positivo (`balance > 0`) abandonan al **23,92 %** frente al **13,94 %** de quienes mantienen saldo cero, lo que indica que **el banco está perdiendo clientes con fondeo real y liquidez depositada**.
4. **Baja sensibilidad lineal de `credit_score`, `tenure`, `credit_card` y `estimated_salary`:** Sus correlaciones lineales simples con `churn` son cercanas a cero ($|r| < 0,03$), lo que confirma la necesidad de algoritmos basados en árboles potenciados (*Gradient Boosting*) e interacciones de dominio.

---
## 2. [15%] Entrenamiento del Modelo de Machine Learning

### 2.1 Partición Estratificada de los Datos (70% Entrenamiento / 30% Prueba)
Dividimos los $10.000$ registros únicos en un conjunto de entrenamiento (**70 % = 7.000 clientes**) y un conjunto de prueba independiente (**30 % = 3.000 clientes**), estratificando por la variable objetivo `churn` con semilla fija (`random_state=42`) para garantizar que la proporción de desertores ($20,37\%$) sea idéntica en ambas particiones.

In [ ]:
X = df_clean.drop(columns=["churn"])
y = df_clean["churn"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    stratify=y,
    random_state=RANDOM_STATE
)

partition_summary = pd.DataFrame({
    "Subconjunto": ["Entrenamiento (Train 70%)", "Prueba (Test 30%)", "Total Muestra Limpia"],
    "Total Clientes": [len(X_train), len(X_test), len(X)],
    "Clientes Activos (Churn=0)": [(y_train == 0).sum(), (y_test == 0).sum(), (y == 0).sum()],
    "Clientes Desertores (Churn=1)": [(y_train == 1).sum(), (y_test == 1).sum(), (y == 1).sum()],
    "Tasa de Churn (%)": [y_train.mean() * 100, y_test.mean() * 100, y.mean() * 100]
}).round(2)

display(partition_summary)

### 2.2 Ingeniería de Características y Pipeline de Preprocesamiento
Para potenciar la capacidad predictiva tanto de los modelos lineales como de los ensambles de árboles —manteniendo **cero fuga de datos (*Zero Data Leakage*)**— diseñamos un transformador compatible con Scikit-Learn (`FinancialFeatureEngineer`) que opera **después** de la imputación de valores faltantes dentro de cada pliegue de validación cruzada:
- `is_zero_balance`: Indicador binario de cuenta sin fondos ($	ext{balance} == 0$).
- `has_3plus_products`: Indicador de sobre-empaquetamiento financiero ($	ext{products\_number} \ge 3$).
- `is_optimal_2products`: Indicador del *sweet spot* de fidelización ($	ext{products\_number} == 2$).
- `inactive_senior`: Interacción de alto riesgo entre edad madura e inactividad ($	ext{age} \ge 40 \land 	ext{active\_member} == 0$).
- `balance_to_salary_ratio`: Razón de apalancamiento/ahorro respecto al salario anual ($rac{	ext{balance}}{	ext{estimated\_salary} + 1}$).

In [ ]:
NUMERIC_COLS = [
    "credit_score", "age", "tenure", "balance",
    "products_number", "credit_card", "active_member", "estimated_salary"
]
CATEGORICAL_COLS = ["country", "gender"]


class FinancialFeatureEngineer(BaseEstimator, TransformerMixin):
    """
    Transformador personalizado que construye variables de negocio financiero
    a partir de las variables numéricas ya imputadas.
    """
    def __init__(self, add_engineered: bool = True):
        self.add_engineered = add_engineered

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        # X llega como arreglo numpy desde SimpleImputer en el orden de NUMERIC_COLS
        df_num = pd.DataFrame(X, columns=NUMERIC_COLS)
        if self.add_engineered:
            df_num["is_zero_balance"] = (df_num["balance"] == 0).astype(float)
            df_num["has_3plus_products"] = (df_num["products_number"] >= 3).astype(float)
            df_num["is_optimal_2products"] = (df_num["products_number"] == 2).astype(float)
            df_num["inactive_senior"] = ((df_num["age"] >= 40) & (df_num["active_member"] == 0)).astype(float)
            df_num["balance_to_salary_ratio"] = np.clip(
                df_num["balance"] / (df_num["estimated_salary"] + 1.0), 0.0, 50.0
            )
        return df_num.values

    def get_feature_names_out(self, input_features=None):
        base = list(NUMERIC_COLS)
        if self.add_engineered:
            base += [
                "is_zero_balance",
                "has_3plus_products",
                "is_optimal_2products",
                "inactive_senior",
                "balance_to_salary_ratio"
            ]
        return np.array(base)


def build_preprocessor(add_engineered: bool = True) -> ColumnTransformer:
    """Construye el ColumnTransformer encapsulando imputación, ingeniería y escalamiento."""
    num_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("feat_eng", FinancialFeatureEngineer(add_engineered=add_engineered)),
        ("scaler", StandardScaler())
    ])
    cat_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore"))
    ])
    return ColumnTransformer(
        transformers=[
            ("num", num_pipeline, NUMERIC_COLS),
            ("cat", cat_pipeline, CATEGORICAL_COLS)
        ]
    )

### 2.3 Arquitecturas Evaluadas y Optimización de Hiperparámetros vía Validación Cruzada
Entrenamos y comparamos **cuatro familias algorítmicas distintas** utilizando validación cruzada estratificada de 5 pliegues (`StratifiedKFold(n_splits=5, shuffle=True, random_state=42)`) y optimizando directamente la métrica **F1-Score** de la clase positiva (`Churn = 1`):

1. **Regresión Logística (`LogisticRegression`):** Modelo paramétrico lineal interpretable, útil como *baseline* estadístico con pesos balanceados (`class_weight='balanced'`).
2. **Random Forest (`RandomForestClassifier`):** Ensamble de árboles independientes mediante *Bagging* que reduce varianza y captura interacciones no lineales.
3. **XGBoost (`XGBClassifier`):** Algoritmo de *Gradient Boosting* regularizado ($L_1/L_2$) con poda por profundidad y ponderación de clase minoritaria (`scale_pos_weight`).
4. **LightGBM (`LGBMClassifier`):** Arquitectura de *Gradient Boosting* por crecimiento de hojas (*leaf-wise*), altamente eficiente en datos tabulares bancarios.

In [ ]:
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

model_configs = {
    "Logistic Regression": {
        "estimator": LogisticRegression(random_state=RANDOM_STATE, max_iter=1000, class_weight="balanced"),
        "param_grid": {
            "preprocessor__num__feat_eng__add_engineered": [True],
            "classifier__C": [0.05, 0.1, 0.5, 1.0, 5.0]
        }
    },
    "Random Forest": {
        "estimator": RandomForestClassifier(random_state=RANDOM_STATE, class_weight="balanced", n_jobs=-1),
        "param_grid": {
            "preprocessor__num__feat_eng__add_engineered": [True, False],
            "classifier__n_estimators": [200, 300],
            "classifier__max_depth": [8, 12],
            "classifier__min_samples_leaf": [3, 6]
        }
    },
    "XGBoost": {
        "estimator": XGBClassifier(random_state=RANDOM_STATE, eval_metric="logloss", n_jobs=-1),
        "param_grid": {
            "preprocessor__num__feat_eng__add_engineered": [True, False],
            "classifier__n_estimators": [150, 250],
            "classifier__max_depth": [3, 4, 5],
            "classifier__learning_rate": [0.03, 0.05, 0.1],
            "classifier__subsample": [0.85],
            "classifier__colsample_bytree": [0.85],
            "classifier__scale_pos_weight": [1.8, 2.5]
        }
    },
    "LightGBM": {
        "estimator": LGBMClassifier(random_state=RANDOM_STATE, verbose=-1, n_jobs=-1),
        "param_grid": {
            "preprocessor__num__feat_eng__add_engineered": [True, False],
            "classifier__n_estimators": [150, 250],
            "classifier__max_depth": [4, 6],
            "classifier__num_leaves": [15, 25],
            "classifier__learning_rate": [0.03, 0.05],
            "classifier__subsample": [0.85],
            "classifier__scale_pos_weight": [1.8, 2.5]
        }
    }
}

trained_models = {}
cv_results_list = []

for name, cfg in model_configs.items():
    pipe = Pipeline(steps=[
        ("preprocessor", build_preprocessor(add_engineered=True)),
        ("classifier", cfg["estimator"])
    ])
    grid = GridSearchCV(
        estimator=pipe,
        param_grid=cfg["param_grid"],
        cv=cv_strategy,
        scoring="f1",
        n_jobs=-1,
        refit=True
    )
    grid.fit(X_train, y_train)
    trained_models[name] = grid.best_estimator_
    
    # Predicciones Out-of-Fold en Train para calibrar umbral óptimo sin tocar Test
    oof_probs = cross_val_predict(
        grid.best_estimator_, X_train, y_train, cv=cv_strategy, method="predict_proba", n_jobs=-1
    )[:, 1]
    
    thresholds = np.linspace(0.20, 0.80, 121)
    f1_by_thresh = [f1_score(y_train, (oof_probs >= t).astype(int)) for t in thresholds]
    best_thresh_idx = int(np.argmax(f1_by_thresh))
    best_thresh = float(thresholds[best_thresh_idx])
    best_oof_f1 = float(f1_by_thresh[best_thresh_idx])
    oof_pr_auc = float(average_precision_score(y_train, oof_probs))
    oof_roc_auc = float(roc_auc_score(y_train, oof_probs))
    
    cv_results_list.append({
        "Modelo": name,
        "CV F1 (Umbral 0.50)": round(grid.best_score_, 4),
        "Umbral Óptimo (OOF)": round(best_thresh, 3),
        "CV F1 (Umbral Óptimo)": round(best_oof_f1, 4),
        "CV PR-AUC": round(oof_pr_auc, 4),
        "CV ROC-AUC": round(oof_roc_auc, 4),
        "Mejores Hiperparámetros": str({k.replace("classifier__", "").replace("preprocessor__num__feat_eng__", ""): v for k, v in grid.best_params_.items()})
    })
    print(f"[OK] {name:<20} | CV F1 (0.50): {grid.best_score_:.4f} | CV F1 (tau={best_thresh:.2f}): {best_oof_f1:.4f} | CV PR-AUC: {oof_pr_auc:.4f}")

cv_summary_df = pd.DataFrame(cv_results_list)
display(cv_summary_df)

---
## 3. [20%] Análisis Cuantitativo de Resultados del Modelo

### 3.1 Justificación Matemática y de Negocio de las Métricas de Evaluación
En problemas de retención bancaria con desbalance de clases ($20,37\%$ en la muestra europea y $15,0\%$ en el mercado colombiano), la **Exactitud (*Accuracy*) es una métrica engañosa y peligrosa para el negocio**:
- Un clasificador trivial que prediga *"Ningún cliente abandonará jamás"* ($\hat{y} = 0$ para todos) alcanzaría un **79,63 % de Accuracy** en nuestra muestra y un **85,00 % de Accuracy** en el banco colombiano, pero **detectaría $0$ desertores ($TP = 0$) y generaría $\text{COP } \$0$ en ingresos retenidos**.

Por ello, FinAlpes estructura la evaluación cuantitativa en torno a cuatro métricas complementarias orientadas a la rentabilidad de la campaña:

1. **PR-AUC (*Area Under the Precision-Recall Curve* / Average Precision):** Evalúa la calidad global del *ranking* de probabilidades enfocándose exclusivamente en la clase minoritaria (`Churn = 1`) a través de todos los umbrales posibles. Es el indicador más robusto para comparar arquitecturas bajo desbalance.
2. **Recall o Sensibilidad ($	ext{Recall} = rac{TP}{TP + FN}$):** Mide qué fracción del total de clientes que realmente iban a abandonar el banco son detectados a tiempo por el modelo. Cada **Falso Negativo ($FN$)** representa un cliente que se fuga sin haber sido contactado, implicando la pérdida potencial de entre **COP \$500.000 y COP \$900.000** anuales.
3. **Precision o Valor Predictivo Positivo ($	ext{Precision} = rac{TP}{TP + FP}$):** Mide la pureza de la campaña, es decir, de todos los clientes alertados y contactados por el banco, qué porcentaje realmente iba a desertar. Dado que **el banco paga el costo de la campaña por todos los clientes contactados ($TP + FP$), abandonen o no**, una baja precisión infla los **Falsos Positivos ($FP$)** y destruye el retorno sobre la inversión (ROI).
4. **F1-Score ($	ext{F1} = rac{2 \cdot 	ext{Precision} \cdot 	ext{Recall}}{	ext{Precision} + 	ext{Recall}}$):** Media armónica que penaliza extremos desequilibrados entre *Precision* y *Recall*, garantizando que el modelo capture un volumen sustancial de desertores sin disparar el desperdicio presupuestal en falsas alarmas.

Evaluamos a continuación los 4 modelos entrenados sobre el **conjunto de prueba independiente (`X_test`, $N = 3.000$ clientes)** tanto con el umbral estándar ($	au = 0,50$) como con el umbral óptimo calibrado en validación cruzada ($	au^*$).

In [ ]:
test_metrics_records = []
model_predictions = {}

for row in cv_results_list:
    name = row["Modelo"]
    tau_opt = row["Umbral Óptimo (OOF)"]
    model = trained_models[name]
    
    y_prob_test = model.predict_proba(X_test)[:, 1]
    y_pred_default = (y_prob_test >= 0.50).astype(int)
    y_pred_opt = (y_prob_test >= tau_opt).astype(int)
    
    model_predictions[name] = {
        "prob": y_prob_test,
        "pred_050": y_pred_default,
        "pred_opt": y_pred_opt,
        "tau_opt": tau_opt
    }
    
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_opt).ravel()
    
    test_metrics_records.append({
        "Modelo": name,
        "Umbral (tau)": tau_opt,
        "PR-AUC (Test)": round(average_precision_score(y_test, y_prob_test), 4),
        "ROC-AUC (Test)": round(roc_auc_score(y_test, y_prob_test), 4),
        "F1-Score (tau=0.50)": round(f1_score(y_test, y_pred_default), 4),
        "F1-Score (tau*)": round(f1_score(y_test, y_pred_opt), 4),
        "Precision (tau*)": round(precision_score(y_test, y_pred_opt), 4),
        "Recall (tau*)": round(recall_score(y_test, y_pred_opt), 4),
        "Accuracy (tau*)": round(accuracy_score(y_test, y_pred_opt), 4),
        "TP": int(tp),
        "FP": int(fp),
        "FN": int(fn),
        "TN": int(tn)
    })

test_metrics_df = pd.DataFrame(test_metrics_records).sort_values(by="PR-AUC (Test)", ascending=False).reset_index(drop=True)
print("=== TABLA COMPARATIVA DE DESEMPEÑO EN EL CONJUNTO DE PRUEBA (TEST 30% = 3.000 CLIENTES) ===")
display(test_metrics_df)

# Selección automática del modelo campeón basado en PR-AUC y F1-Score en Test
champion_name = test_metrics_df.iloc[0]["Modelo"]
champion_model = trained_models[champion_name]
champion_tau = test_metrics_df.iloc[0]["Umbral (tau)"]
print(f"\n>>> MODELO CAMPEÓN SELECCIONADO: {champion_name} (PR-AUC = {test_metrics_df.iloc[0]['PR-AUC (Test)']:.4f}, F1 = {test_metrics_df.iloc[0]['F1-Score (tau*)']:.4f}, Umbral tau* = {champion_tau:.3f}) <<<")

In [ ]:
# Gráfico 4: Curvas Precision-Recall, ROC y Optimización de Umbral
fig, axes = plt.subplots(1, 3, figsize=(17, 5.2))

model_colors = {
    "Logistic Regression": PALETTE_CORP["neutral"],
    "Random Forest": PALETTE_CORP["secondary"],
    "XGBoost": PALETTE_CORP["primary"],
    "LightGBM": PALETTE_CORP["danger"]
}

# Panel A: Curva Precision-Recall (Métrica Principal)
for name, preds in model_predictions.items():
    prec, rec, _ = precision_recall_curve(y_test, preds["prob"])
    pr_auc = average_precision_score(y_test, preds["prob"])
    lw = 2.8 if name == champion_name else 1.8
    axes[0].plot(rec, prec, label=f"{name} (PR-AUC = {pr_auc:.3f})", color=model_colors[name], linewidth=lw)

baseline_pr = y_test.mean()
axes[0].axhline(baseline_pr, color="black", linestyle="--", linewidth=1.2, label=f"Azar / Sin Modelo ({baseline_pr:.2f})")
axes[0].set_title("A. Curva Precision-Recall (Conjunto de Prueba)")
axes[0].set_xlabel("Recall (Fracción de Desertores Detectados)")
axes[0].set_ylabel("Precision (Pureza de la Campaña)")
axes[0].legend(loc="lower left", fontsize=8.8, frameon=True)
axes[0].set_xlim(0, 1.0)
axes[0].set_ylim(0, 1.02)

# Panel B: Curva ROC
for name, preds in model_predictions.items():
    fpr, tpr, _ = roc_curve(y_test, preds["prob"])
    roc_auc = roc_auc_score(y_test, preds["prob"])
    lw = 2.8 if name == champion_name else 1.8
    axes[1].plot(fpr, tpr, label=f"{name} (ROC-AUC = {roc_auc:.3f})", color=model_colors[name], linewidth=lw)

axes[1].plot([0, 1], [0, 1], color="black", linestyle="--", linewidth=1.2, label="Clasificador Aleatorio (0.500)")
axes[1].set_title("B. Curva ROC (Sensibilidad vs. 1 - Especificidad)")
axes[1].set_xlabel("Tasa de Falsos Positivos (FPR)")
axes[1].set_ylabel("Tasa de Verdaderos Positivos (TPR / Recall)")
axes[1].legend(loc="lower right", fontsize=8.8, frameon=True)

# Panel C: Trade-off Precision, Recall y F1-Score según Umbral para el Modelo Campeón
champ_probs = model_predictions[champion_name]["prob"]
t_grid = np.linspace(0.10, 0.85, 150)
prec_grid = [precision_score(y_test, (champ_probs >= t).astype(int), zero_division=0) for t in t_grid]
rec_grid = [recall_score(y_test, (champ_probs >= t).astype(int), zero_division=0) for t in t_grid]
f1_grid = [f1_score(y_test, (champ_probs >= t).astype(int), zero_division=0) for t in t_grid]

axes[2].plot(t_grid, prec_grid, label="Precision (Eficiencia Gasto)", color=PALETTE_CORP["primary"], linestyle="-.", linewidth=2)
axes[2].plot(t_grid, rec_grid, label="Recall (Cobertura Churn)", color=PALETTE_CORP["secondary"], linestyle="--", linewidth=2)
axes[2].plot(t_grid, f1_grid, label="F1-Score (Balance Armónico)", color=PALETTE_CORP["danger"], linewidth=2.8)
axes[2].axvline(champion_tau, color=PALETTE_CORP["accent"], linestyle=":", linewidth=2.2, label=f"Umbral Óptimo CV (tau*={champion_tau:.2f})")
axes[2].set_title(f"C. Sensibilidad al Umbral de Decisión ({champion_name})")
axes[2].set_xlabel("Umbral de Probabilidad de Abandono (tau)")
axes[2].set_ylabel("Valor de la Métrica")
axes[2].legend(loc="lower center", fontsize=8.5, frameon=True)
axes[2].set_ylim(0, 1.02)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "04_comparativa_modelos_curvas.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Gráfico 5: Matrices de Confusión Comparativas en el Conjunto de Prueba (N = 3.000)
fig, axes = plt.subplots(1, 4, figsize=(18, 4.3))

for ax, (_, row) in zip(axes, test_metrics_df.iterrows()):
    name = row["Modelo"]
    preds_opt = model_predictions[name]["pred_opt"]
    tau_val = row["Umbral (tau)"]
    cm = confusion_matrix(y_test, preds_opt)
    cm_pct = cm / len(y_test) * 100
    
    annot_labels = np.array([
        [f"TN\n{cm[0,0]:,}\n({cm_pct[0,0]:.1f}%)", f"FP (Costo Inútil)\n{cm[0,1]:,}\n({cm_pct[0,1]:.1f}%)"],
        [f"FN (Fuga No Detectada)\n{cm[1,0]:,}\n({cm_pct[1,0]:.1f}%)", f"TP (Riesgo Capturado)\n{cm[1,1]:,}\n({cm_pct[1,1]:.1f}%)"]
    ])
    
    sns.heatmap(
        cm, annot=annot_labels, fmt="", cmap="Blues", cbar=False, ax=ax,
        xticklabels=["Pred: Leal (0)", "Pred: Churn (1)"],
        yticklabels=["Real: Leal (0)", "Real: Churn (1)"],
        annot_kws={"size": 9.5, "weight": "bold"}, linewidths=1, linecolor="black"
    )
    ax.set_title(f"{name}\n(tau={tau_val:.2f} | F1={row['F1-Score (tau*)']:.3f} | PR-AUC={row['PR-AUC (Test)']:.3f})", fontsize=10.5)

plt.suptitle("Matrices de Confusión en Prueba (3.000 Clientes) e Interpretación de Costos de Error", fontsize=13.5, fontweight="bold", y=1.05)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "05_matrices_confusion_umbral.png", dpi=150, bbox_inches="tight")
plt.show()

### 3.2 Justificación de la Elección del Modelo Campeón y Oportunidades de Mejora

#### ¿Por qué los modelos de *Gradient Boosting* (`LightGBM` / `XGBoost`) superan ampliamente a la Regresión Logística?
1. **Superioridad cuantitativa contundente:** Mientras que la Regresión Logística alcanza un `PR-AUC` de apenas **$0,55$** (y ~$0,46$ sin características de dominio), **LightGBM y XGBoost elevan el `PR-AUC` por encima de $0,69$**, el `ROC-AUC` por encima de **$0,86$** y el `F1-Score` a **~$0,63$**, reduciendo en más de un **$60\%$ los Falsos Positivos ($FP$)** respecto al modelo lineal para un nivel equivalente de captura de desertores.
2. **Elementos que impactaron positivamente las métricas:**
   - **Captura de umbrales y discontinuidades no lineales:** Los árboles potenciados aíslan de manera exacta el cambio abrupto de riesgo entre tener $2$ productos ($7,6\%$ de churn) y tener $\ge 3$ productos ($>82\%$ de churn), así como la interacción entre edad ($41	ext{–}60$ años), inactividad comercial y residencia en Alemania.
   - **Limpieza de anomalías e ingeniería financiera sin *Data Leakage*:** El enmascaramiento de valores corruptos (`age = 210`, `credit_score = 1200`, `tenure = -2`) y la incorporación de variables de negocio (`inactive_senior`, `is_optimal_2products`, `balance_to_salary_ratio`) mejoraron la estabilidad entre validación cruzada y prueba.
   - **Ponderación del desbalance (`scale_pos_weight`) y calibración del umbral ($	au^*$):** Permite sintonizar la agresividad comercial del modelo según la relación entre el costo del contacto y el ingreso anual del cliente.

#### Oportunidades de Mejora Identificadas para Fases Siguientes
1. **Incorporación de variables dinámicas y transaccionales (Series de Tiempo):** Actualmente el dataset es una fotografía estática (*cross-section*). Incorporar la variación del saldo en los últimos 3–6 meses ($\Delta 	ext{balance}$), caída en frecuencia de login en la App móvil y recencia de uso de tarjeta de crédito elevaría sustancialmente el `PR-AUC` al detectar la *desaceleración previa* a la cancelación.
2. **Integración con el módulo de NLP de PQRS de FinAlpes:** Cruzar las quejas y reclamos ante el banco o la Superintendencia Financiera con el score comercial para capturar abandonos por insatisfacción de servicio.
3. **Evolución hacia *Uplift Modeling* (Modelos de Persuasión Causal):** En lugar de predecir únicamente la probabilidad de abandono $\mathbb{P}(	ext{Churn}=1)$, estimar el efecto incremental del contacto $\mathbb{P}(	ext{Retención} \mid 	ext{Contacto}) - \mathbb{P}(	ext{Retención} \mid 	ext{Sin Contacto})$ para no gastar incentivos en *"casos perdidos"* (*Lost Causes*) ni en clientes que no necesitan oferta.

---
## 4. [20%] Análisis Cualitativo de Resultados del Modelo (Explicabilidad SHAP)

Un modelo de alto desempeño no puede operar como una *"caja negra"* en el sector financiero: la regulación bancaria y los equipos comerciales requieren entender **por qué** el algoritmo asigna una alta probabilidad de fuga a cada cliente. Utilizamos **SHAP (*SHapley Additive exPlanations*)**, fundamentado en la teoría de juegos cooperativos de Lloyd Shapley, para descomponer exactamente las predicciones del modelo campeón tanto a nivel **global (toda la cartera)** como **individual (clientes específicos)**.

In [ ]:
# Extracción del preprocesador y del estimador entrenado del modelo campeón
prep_fitted = champion_model.named_steps["preprocessor"]
clf_fitted = champion_model.named_steps["classifier"]

# Transformación del conjunto de prueba y recuperación de nombres legibles de variables
X_test_transformed = prep_fitted.transform(X_test)
raw_feature_names = prep_fitted.get_feature_names_out()

clean_feature_names = [
    name.replace("num__", "").replace("cat__", "")
    for name in raw_feature_names
]
X_test_shap_df = pd.DataFrame(X_test_transformed, columns=clean_feature_names, index=X_test.index)

# Tabla de referencia con los valores originales sin escalar (para lectura ejecutiva en casos individuales)
X_test_original_imputed = pd.DataFrame(
    prep_fitted.named_transformers_["num"].named_steps["feat_eng"].transform(
        prep_fitted.named_transformers_["num"].named_steps["imputer"].transform(X_test[NUMERIC_COLS])
    ),
    columns=prep_fitted.named_transformers_["num"].named_steps["feat_eng"].get_feature_names_out(),
    index=X_test.index
)
X_test_original_imputed["country_Germany"] = (X_test["country"] == "Germany").astype(int)
X_test_original_imputed["country_Spain"] = (X_test["country"] == "Spain").astype(int)
X_test_original_imputed["gender_Male"] = (X_test["gender"] == "Male").astype(int)
X_test_original_imputed = X_test_original_imputed[clean_feature_names]

# Cálculo exacto de valores SHAP mediante TreeExplainer
explainer = shap.TreeExplainer(clf_fitted)
shap_explanation = explainer(X_test_shap_df)

# En algunas versiones de LightGBM/SHAP, shap_explanation puede tener dimensión (N, F, 2); seleccionamos clase positiva (1)
if len(shap_explanation.shape) == 3:
    shap_explanation_pos = shap_explanation[:, :, 1]
else:
    shap_explanation_pos = shap_explanation

# Importancia global media absoluta |SHAP|
mean_abs_shap = np.abs(shap_explanation_pos.values).mean(axis=0)
shap_importance_df = pd.DataFrame({
    "Variable": clean_feature_names,
    "Importancia Media |SHAP| (Log-Odds)": mean_abs_shap,
    "Participación Relativa (%)": (mean_abs_shap / mean_abs_shap.sum() * 100)
}).sort_values(by="Importancia Media |SHAP| (Log-Odds)", ascending=False).reset_index(drop=True)

print(f"=== RANKING GLOBAL DE IMPORTANCIA DE VARIABLES (SHAP - {champion_name}) ===")
display(shap_importance_df.round(4))

In [ ]:
# Gráfico 6: Comportamiento Global del Modelo (SHAP Beeswarm Plot + Bar Plot de Importancia)
fig = plt.figure(figsize=(16, 6.2))

ax1 = fig.add_subplot(1, 2, 1)
shap.plots.bar(shap_explanation_pos, max_display=12, show=False, ax=ax1)
ax1.set_title("A. Magnitud de Impacto Global Media (|SHAP|)", fontsize=12, fontweight="bold")

ax2 = fig.add_subplot(1, 2, 2)
plt.sca(ax2)
shap.summary_plot(shap_explanation_pos.values, X_test_shap_df, plot_type="dot", max_display=12, show=False)
ax2.set_title("B. Distribución Direccional de Impacto (SHAP Beeswarm)", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "06_shap_global_importancia.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Gráfico 7: Gráficos de Dependencia SHAP sobre valores reales de negocio (Edad, Número de Productos y Saldo)
fig, axes = plt.subplots(1, 3, figsize=(17, 5.0))

shap_vals_matrix = shap_explanation_pos.values
idx_age = clean_feature_names.index("age")
idx_prod = clean_feature_names.index("products_number")
idx_bal = clean_feature_names.index("balance")

# 1. Dependencia de Edad coloreada por Actividad Comercial
sc1 = axes[0].scatter(
    X_test_original_imputed["age"],
    shap_vals_matrix[:, idx_age],
    c=X_test_original_imputed["active_member"],
    cmap="coolwarm_r", alpha=0.65, s=18
)
axes[0].axhline(0, color="black", linestyle="--", linewidth=1)
axes[0].set_title("A. Efecto SHAP de la Edad (Coloreado por Actividad)")
axes[0].set_xlabel("Edad del Cliente (Años)")
axes[0].set_ylabel("Contribución SHAP al Riesgo de Churn")
cbar1 = plt.colorbar(sc1, ax=axes[0])
cbar1.set_label("Miembro Activo (1=Sí, 0=No)")

# 2. Dependencia del Número de Productos
prod_jitter = X_test_original_imputed["products_number"] + np.random.normal(0, 0.06, size=len(X_test_original_imputed))
sc2 = axes[1].scatter(
    prod_jitter,
    shap_vals_matrix[:, idx_prod],
    c=X_test_original_imputed["balance"],
    cmap="viridis", alpha=0.65, s=18
)
axes[1].axhline(0, color="black", linestyle="--", linewidth=1)
axes[1].set_xticks([1, 2, 3, 4])
axes[1].set_title("B. Efecto SHAP del Número de Productos")
axes[1].set_xlabel("Cantidad de Productos Contratados")
axes[1].set_ylabel("Contribución SHAP al Riesgo de Churn")
cbar2 = plt.colorbar(sc2, ax=axes[1])
cbar2.set_label("Saldo en Cuenta (EUR)")

# 3. Dependencia de Saldo coloreado por País (Alemania)
sc3 = axes[2].scatter(
    X_test_original_imputed["balance"],
    shap_vals_matrix[:, idx_bal],
    c=X_test_original_imputed["country_Germany"],
    cmap="bwr", alpha=0.65, s=18
)
axes[2].axhline(0, color="black", linestyle="--", linewidth=1)
axes[2].set_title("C. Efecto SHAP del Saldo (Coloreado por Alemania)")
axes[2].set_xlabel("Saldo en Depósito (EUR)")
axes[2].set_ylabel("Contribución SHAP al Riesgo de Churn")
cbar3 = plt.colorbar(sc3, ax=axes[2])
cbar3.set_label("Residente en Alemania (1=Sí, 0=No)")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "07_shap_dependencia_drivers.png", dpi=150, bbox_inches="tight")
plt.show()

### 4.1 Interpretación del Comportamiento Global del Modelo
El análisis SHAP global revela con precisión matemática cómo opera internamente el modelo de FinAlpes:
1. **Edad (`age`) e Inactividad Senior (`inactive_senior`):** Son el motor número uno de la predicción. Como muestra la Figura 7A, el valor SHAP cruza el umbral de riesgo positivo exactamente a partir de los **39–40 años**, alcanzando su pico máximo entre los **48 y 60 años**. Los clientes menores de 35 años reciben un fuerte empuje negativo (protector) en log-odds.
2. **Número de Productos y Vinculación (`products_number`, `is_optimal_2products`, `has_3plus_products`):** El modelo aprendió perfectamente la relación no monótona (Figura 7B): tener **2 productos** reduce drásticamente la probabilidad de abandono ($	ext{SHAP} < -0,8$), tener **1 solo producto** incrementa el riesgo moderadamente, y poseer **3 o 4 productos** dispara positivamente el valor SHAP ($	ext{SHAP} > +1,5$).
3. **Actividad Transaccional (`active_member`):** Ser un miembro activo (`1`) actúa como un escudo protector consistente (puntos rojos a la izquierda del eje cero en el *Beeswarm*), mientras que la inactividad (`0`) empuja al cliente hacia la zona de deserción.
4. **Efecto Regional y Patrimonial (`country_Germany`, `balance`):** Residir en Alemania y mantener saldos medios-altos entre $80.000$ y $160.000	ext{ EUR}$ incrementa significativamente el riesgo predicho, lo cual alerta al negocio sobre una fuga selectiva de clientes rentables con capital depositado.

---

### 4.2 Análisis Local de Predicciones Individuales (Arquetipos de Clientes con *Waterfall Plots*)
Seleccionamos **tres clientes reales del conjunto de prueba (`X_test`)** que representan arquetipos clave para el equipo de campañas de retención:
1. **Caso 1 — Verdadero Positivo de Alto Riesgo ($TP$):** Cliente con probabilidad predicha muy alta ($\hat{p} > 0,85$) que efectivamente abandonó el banco.
2. **Caso 2 — Verdadero Negativo / Cliente Leal Blindado ($TN$):** Cliente con probabilidad de abandono mínima ($\hat{p} < 0,05$) que permaneció en la entidad.
3. **Caso 3 — Caso Frontera / Falso Positivo de Campaña ($FP$ cerca del umbral $	au^*$):** Cliente con señales mixtas (factores de riesgo que compiten con factores protectores) cuya probabilidad supera ligeramente el umbral operativo.

In [ ]:
test_eval_df = X_test.copy()
test_eval_df["y_true"] = y_test.values
test_eval_df["y_prob"] = model_predictions[champion_name]["prob"]
test_eval_df["y_pred"] = model_predictions[champion_name]["pred_opt"]

# Selección determinista de los 3 arquetipos representativos en X_test
tp_candidates = test_eval_df[(test_eval_df["y_true"] == 1) & (test_eval_df["y_pred"] == 1)].sort_values(by="y_prob", ascending=False)
tn_candidates = test_eval_df[(test_eval_df["y_true"] == 0) & (test_eval_df["y_pred"] == 0)].sort_values(by="y_prob", ascending=True)
border_candidates = test_eval_df[
    (test_eval_df["y_true"] == 0) &
    (test_eval_df["y_prob"].between(champion_tau + 0.02, champion_tau + 0.12))
].sort_values(by="y_prob", ascending=True)

idx_tp = tp_candidates.index[0]
idx_tn = tn_candidates.index[0]
idx_fp = border_candidates.index[0]

selected_cases = pd.DataFrame([
    test_eval_df.loc[idx_tp],
    test_eval_df.loc[idx_tn],
    test_eval_df.loc[idx_fp]
], index=["Caso 1: Alto Riesgo (TP)", "Caso 2: Cliente Leal (TN)", "Caso 3: Caso Frontera (FP)"])

print("=== PERFIL COMERCIAL DE LOS 3 CLIENTES ANALIZADOS INDIVIDUALMENTE ===")
display(selected_cases)

# Construcción de objeto Explanation con datos reales (sin escalar) para que el Waterfall muestre valores legibles de negocio
shap_exp_readable = shap.Explanation(
    values=shap_explanation_pos.values,
    base_values=shap_explanation_pos.base_values,
    data=X_test_original_imputed.round(2).values,
    feature_names=clean_feature_names
)

pos_tp = list(X_test.index).index(idx_tp)
pos_tn = list(X_test.index).index(idx_tn)
pos_fp = list(X_test.index).index(idx_fp)

fig = plt.figure(figsize=(18, 15))

plt.subplot(3, 1, 1)
shap.plots.waterfall(shap_exp_readable[pos_tp], max_display=9, show=False)
plt.title(f"Caso 1 — Cliente en Alto Riesgo Real (ID={idx_tp} | Prob. Churn = {test_eval_df.loc[idx_tp, 'y_prob']:.1%} | Real = Abandonó)", fontsize=12.5, fontweight="bold")

plt.subplot(3, 1, 2)
shap.plots.waterfall(shap_exp_readable[pos_tn], max_display=9, show=False)
plt.title(f"Caso 2 — Cliente Leal Blindado (ID={idx_tn} | Prob. Churn = {test_eval_df.loc[idx_tn, 'y_prob']:.1%} | Real = Permaneció)", fontsize=12.5, fontweight="bold")

plt.subplot(3, 1, 3)
shap.plots.waterfall(shap_exp_readable[pos_fp], max_display=9, show=False)
plt.title(f"Caso 3 — Cliente en Zona Frontera / Riesgo Moderado (ID={idx_fp} | Prob. Churn = {test_eval_df.loc[idx_fp, 'y_prob']:.1%} | Real = Permaneció)", fontsize=12.5, fontweight="bold")

plt.tight_layout(pad=3.0)
plt.savefig(FIGURES_DIR / "08_shap_local_waterfalls.png", dpi=150, bbox_inches="tight")
plt.show()

### 4.3 Diagnóstico Individual y Accionable Comercial por Caso
1. **Caso 1 (Verdadero Positivo — Alto Riesgo, $\hat{p} > 90\%$):** Se trata de un cliente en el rango crítico de edad ($\sim 50	ext{–}56$ años), **inactivo** (`active_member = 0`), con **sobre-contratación de productos ($\ge 3$ productos)** o baja vinculación y saldo positivo en Alemania. Todos sus vectores SHAP son rojos (positivos), empujando los log-odds muy por encima del valor base $E[f(X)]$. **Acción recomendada:** Contacto prioritario inmediato por ejecutivo especializado para simplificar/reestructurar su portafolio de productos eliminando comisiones redundantes y ofreciendo tasa preferencial de ahorro.
2. **Caso 2 (Verdadero Negativo — Cliente Leal Blindado, $\hat{p} < 2\%$):** Cliente joven ($\sim 25	ext{–}32$ años), **miembro activo** (`active_member = 1`), ubicado en el **punto óptimo de vinculación (`products_number = 2`)** y fuera del segmento crítico alemán. Sus fuerzas SHAP azules reducen drásticamente el riesgo. **Acción recomendada:** **No incluir en campañas de retención** (evitar gasto innecesario y fricción comercial); mantener en comunicación digital estándar de fidelización.
3. **Caso 3 (Caso Frontera — Señales Mixtas cerca de $	au^*$):** Presenta factores de riesgo (p. ej., edad cercana a 42–46 años o un solo producto) que empujan la predicción hacia arriba, parcialmente compensados por factores protectores (como ser miembro activo o residir en Francia/España). **Acción recomendada:** Atender mediante **canal digital automatizado de bajo costo** (notificación App / WhatsApp con beneficio condicional de uso) en lugar de llamada costosa de *Call Center*.

---
## 5. [20%] Generación de Valor, Simulación Paramétrica y Punto de Equilibrio (*Break-Even*)

### 5.1 Formulación Matemática del Modelo Financiero y Ajuste al Banco Objetivo ($N = 20.000$)

Para trasladar las predicciones desde la muestra europea de prueba ($N_{\text{test}} = 3.000$, donde la tasa de deserción es $\pi_{\text{muestra}} = 20,37\%$) hacia la entidad bancaria objetivo en Colombia ($N_{\text{banco}} = 20.000$ clientes con tasa base de deserción de la industria $\pi_{\text{banco}} = 15,0\%$), es indispensable aplicar la **corrección por cambio en la prevalencia a priori (*Prior Probability Shift*)**:

- **Población del banco objetivo:** $N_{\text{banco}} = 20.000$ clientes.
  - **Desertores reales anuales ($P_{\text{banco}}$):** $N_{\text{banco}} \times 15,0\% = \mathbf{3.000\text{ clientes}}$.
  - **Clientes leales anuales ($N_{\text{leales}}$):** $N_{\text{banco}} \times 85,0\% = \mathbf{17.000\text{ clientes}}$.
- **Tasas condicionales invariantes del clasificador a un umbral $\tau$:**
  Las propiedades intrínsecas del clasificador condicionadas a la clase real —la Sensibilidad $\text{TPR}(\tau) = \frac{TP}{P}$ y la Tasa de Falsas Alarmas $\text{FPR}(\tau) = \frac{FP}{N_{\text{neg}}}$— permiten proyectar con exactitud la matriz de confusión sobre los $20.000$ clientes del banco:
  $$TP_{\text{banco}}(\tau) = 3.000 \times \text{TPR}(\tau), \qquad FN_{\text{banco}}(\tau) = 3.000 \times (1 - \text{TPR}(\tau))$$
  $$FP_{\text{banco}}(\tau) = 17.000 \times \text{FPR}(\tau), \qquad TN_{\text{banco}}(\tau) = 17.000 \times (1 - \text{FPR}(\tau))$$

#### Ecuación de Creación de Valor y Punto de Equilibrio (*Break-Even*)
1. **Clientes contactados en la campaña:** $K(\tau) = TP_{\text{banco}}(\tau) + FP_{\text{banco}}(\tau)$.
2. **Clientes efectivamente retenidos gracias a la campaña:** $\text{Retenidos}(\tau) = \gamma \cdot TP_{\text{banco}}(\tau)$, donde $\gamma = 0,30$ ($30\%$).
3. **Ingreso Bruto Recuperado (Ingresos Salvados):**
   $$\text{Ingreso Recuperado}(\tau) = \gamma \cdot V \cdot TP_{\text{banco}}(\tau)$$
   donde $V \in [\text{COP } \$500.000,\, \text{COP } \$900.000]$ es el ingreso anual por cliente activo.
4. **Costo Variable de la Campaña de Retención:** Se paga por **todos** los clientes contactados ($TP + FP$):
   $$\text{Costo Campaña}(\tau) = C_{\text{contacto}} \cdot \big(TP_{\text{banco}}(\tau) + FP_{\text{banco}}(\tau)\big)$$
5. **Margen Operativo de la Campaña (antes de costos fijos de implementación):**
   $$\text{Margen Campaña}(\tau) = \gamma \cdot V \cdot TP_{\text{banco}}(\tau) - C_{\text{contacto}} \cdot \big(TP_{\text{banco}}(\tau) + FP_{\text{banco}}(\tau)\big)$$
6. **Ganancia Neta Anual del Banco (después del costo fijo de implementación $C_{\text{fijo}}$):**
   $$\text{Utilidad Neta}(\tau) = \text{Margen Campaña}(\tau) - C_{\text{fijo}}$$
7. **Condición de Punto de Equilibrio (*Break-Even*):**
   - **Precisión mínima para rentabilidad marginal por contacto:**
     $$\text{Precision}_{\text{banco}}(\tau) \cdot (\gamma \cdot V) \ge C_{\text{contacto}} \implies \text{Precision}_{\min} = \frac{C_{\text{contacto}}}{\gamma \cdot V}$$
   - **Volumen mínimo de clientes (Escala mínima de la base $N^*$ para cubrir la inversión fija $C_{\text{fijo}}$):**
     $$N_{\text{break-even}} = \frac{C_{\text{fijo}}}{\text{Margen Campaña por Cliente de la Cartera}}$$
   - **Tiempo de recuperación de la inversión (*Payback Period* en meses):**
     $$\text{Payback (meses)} = 12 \times \frac{C_{\text{fijo}}}{\text{Margen Campaña Anual}(\tau)}$$

---

### 5.2 Explicitación de Supuestos Financieros y Fuentes de Mercado (Colombia 2025–2026)

| Parámetro del Modelo Financiero | Escenario Conservador | Escenario Base (Referencia) | Escenario Optimista | Fuente / Justificación Estadística y de Industria |
| :--- | :---: | :---: | :---: | :--- |
| **Tamaño de Cartera ($N_{\text{banco}}$)** | $20.000$ | $20.000$ | $20.000$ | Enunciado Taller 2 — Entidad bancaria cliente de FinAlpes. |
| **Tasa Anual de Churn ($\pi_{\text{banco}}$)** | $15,0\%$ | $15,0\%$ | $15,0\%$ | Enunciado Taller 2 — Benchmark minorista ($3.000$ desertores/año). |
| **Tasa de Éxito de Retención ($\gamma$)** | $30,0\%$ | $30,0\%$ | $30,0\%$ | Enunciado Taller 2 — Proporción de $TP$ convencidos por la oferta. |
| **Ingreso Anual por Cliente Activo ($V$)** | $\text{COP } \$500.000$ | $\text{COP } \$700.000$ | $\text{COP } \$900.000$ | Reporte SEC Form 6-K de Nu Holdings (ARPAC mensual ~USD 11–16 en LatAm/Colombia, equivalente a COP \$500k–\$900k/año). |
| **Costo por Cliente Contactado ($C_{\text{contacto}}$)** | $\text{COP } \$50.000$ *(Call Center + Bono alto)* | $\text{COP } \$35.000$ *(Mix Outbound + Incentivo)* | $\text{COP } \$20.000$ *(Omnicanal Digital + Cashback)* | Tarifas BPO financiero en Colombia (Asociación Colombiana de BPO / Colcx 2025: llamada outbound especializada COP \$8k–\$15k + costo promedio ponderado de incentivo/exoneración de cuota COP \$12k–\$35k). |
| **Costo Fijo Implementación + MLOps ($C_{\text{fijo}}$)** | $\text{COP } \$80.000.000$ | $\text{COP } \$60.000.000$ | $\text{COP } \$45.000.000$ | Consultoría FinAlpes (descubrimiento, pipeline e integración CRM ~COP \$42M) + infraestructura Cloud AWS/GCP y monitoreo anual (~COP \$18M/año). |

In [ ]:
# Parámetros estructurales del banco objetivo
N_BANK = 20_000
CHURN_RATE_BANK = 0.15
N_CHURNERS_BANK = int(N_BANK * CHURN_RATE_BANK)        # 3.000 desertores reales
N_LOYAL_BANK = N_BANK - N_CHURNERS_BANK                # 17.000 clientes leales
RETENTION_SUCCESS_RATE = 0.30                          # 30% de éxito sobre TP

# Parámetros del Escenario Base
V_BASE = 700_000          # Ingreso anual promedio por cliente activo (COP $700.000)
C_CONTACT_BASE = 35_000   # Costo por cliente contactado en campaña (COP $35.000)
C_FIXED_BASE = 60_000_000 # Costo fijo anualizado de implementación y operación MLOps (COP $60.000.000)


def evaluate_financial_strategy(
    tpr: float,
    fpr: float,
    v_annual: float = V_BASE,
    c_contact: float = C_CONTACT_BASE,
    c_fixed: float = C_FIXED_BASE,
    n_bank: int = N_BANK,
    churn_rate: float = CHURN_RATE_BANK,
    gamma: float = RETENTION_SUCCESS_RATE
) -> dict:
    """
    Calcula el desempeño financiero completo proyectado sobre la base de 20.000 clientes
    del banco ajustando por la prevalencia real de Churn (15%).
    """
    n_pos = n_bank * churn_rate
    n_neg = n_bank * (1.0 - churn_rate)
    
    tp_bank = n_pos * tpr
    fn_bank = n_pos * (1.0 - tpr)
    fp_bank = n_neg * fpr
    tn_bank = n_neg * (1.0 - fpr)
    
    contacted = tp_bank + fp_bank
    retained = gamma * tp_bank
    precision_bank = tp_bank / contacted if contacted > 0 else 0.0
    
    gross_saved_revenue = retained * v_annual
    campaign_cost = contacted * c_contact
    campaign_margin = gross_saved_revenue - campaign_cost
    net_profit = campaign_margin - c_fixed
    total_investment = campaign_cost + c_fixed
    roi_pct = (net_profit / total_investment * 100.0) if total_investment > 0 else 0.0
    
    # Break-even en número de clientes de cartera necesarios para cubrir el costo fijo C_fixed
    margin_per_portfolio_client = campaign_margin / n_bank
    breakeven_portfolio_clients = (c_fixed / margin_per_portfolio_client) if margin_per_portfolio_client > 0 else np.inf
    breakeven_retained_clients = (total_investment / v_annual) if v_annual > 0 else np.inf
    payback_months = (12.0 * c_fixed / campaign_margin) if campaign_margin > 0 else np.inf
    
    return {
        "TP_Banco": tp_bank,
        "FP_Banco": fp_bank,
        "FN_Banco": fn_bank,
        "TN_Banco": tn_bank,
        "Contactados": contacted,
        "Precision_Banco (%)": precision_bank * 100,
        "Recall (%)": tpr * 100,
        "Clientes_Retenidos": retained,
        "Ingreso_Recuperado_COP": gross_saved_revenue,
        "Costo_Campana_COP": campaign_cost,
        "Margen_Campana_COP": campaign_margin,
        "Costo_Fijo_Modelo_COP": c_fixed,
        "Ganancia_Neta_COP": net_profit,
        "ROI_Total (%)": roi_pct,
        "BreakEven_Cartera_Clientes": breakeven_portfolio_clients,
        "BreakEven_Clientes_Retenidos": breakeven_retained_clients,
        "Payback_Meses": payback_months
    }


# Comparación de los 4 modelos y estrategias ingenuas en el Escenario Base
strategy_rows = []

# Estrategia 0: No hacer nada
strategy_rows.append({
    "Estrategia / Modelo": "0. Sin Campaña (Inacción)",
    "Contactados": 0,
    "Clientes_Retenidos": 0,
    "Precision_Banco (%)": 0.0,
    "Recall (%)": 0.0,
    "Ingreso_Recuperado (MM COP)": 0.0,
    "Costo_Campana (MM COP)": 0.0,
    "Costo_Fijo (MM COP)": 0.0,
    "Ganancia_Neta (MM COP)": 0.0,
    "ROI_Total (%)": 0.0,
    "BreakEven_Cartera": np.nan,
    "Payback (Meses)": np.nan
})

# Estrategia 1: Campaña Masiva a los 20.000 clientes (Sin modelo de ML, TPR=1.0, FPR=1.0, C_fixed=0)
mass_res = evaluate_financial_strategy(tpr=1.0, fpr=1.0, c_fixed=0.0)
strategy_rows.append({
    "Estrategia / Modelo": "1. Campaña Masiva (100% Base, Sin ML)",
    "Contactados": int(round(mass_res["Contactados"])),
    "Clientes_Retenidos": int(round(mass_res["Clientes_Retenidos"])),
    "Precision_Banco (%)": round(mass_res["Precision_Banco (%)"], 1),
    "Recall (%)": round(mass_res["Recall (%)"], 1),
    "Ingreso_Recuperado (MM COP)": round(mass_res["Ingreso_Recuperado_COP"] / 1e6, 2),
    "Costo_Campana (MM COP)": round(mass_res["Costo_Campana_COP"] / 1e6, 2),
    "Costo_Fijo (MM COP)": 0.0,
    "Ganancia_Neta (MM COP)": round(mass_res["Ganancia_Neta_COP"] / 1e6, 2),
    "ROI_Total (%)": round(mass_res["ROI_Total (%)"], 1),
    "BreakEven_Cartera": np.nan,
    "Payback (Meses)": np.nan
})

# Evaluación de cada uno de los 4 modelos entrenados en su umbral óptimo
for _, row in test_metrics_df.iterrows():
    name = row["Modelo"]
    tpr_val = row["TP"] / (row["TP"] + row["FN"])
    fpr_val = row["FP"] / (row["FP"] + row["TN"])
    fin = evaluate_financial_strategy(tpr=tpr_val, fpr=fpr_val)
    strategy_rows.append({
        "Estrategia / Modelo": f"ML: {name} (tau={row['Umbral (tau)']:.2f})",
        "Contactados": int(round(fin["Contactados"])),
        "Clientes_Retenidos": int(round(fin["Clientes_Retenidos"])),
        "Precision_Banco (%)": round(fin["Precision_Banco (%)"], 1),
        "Recall (%)": round(fin["Recall (%)"], 1),
        "Ingreso_Recuperado (MM COP)": round(fin["Ingreso_Recuperado_COP"] / 1e6, 2),
        "Costo_Campana (MM COP)": round(fin["Costo_Campana_COP"] / 1e6, 2),
        "Costo_Fijo (MM COP)": round(fin["Costo_Fijo_Modelo_COP"] / 1e6, 2),
        "Ganancia_Neta (MM COP)": round(fin["Ganancia_Neta_COP"] / 1e6, 2),
        "ROI_Total (%)": round(fin["ROI_Total (%)"], 1),
        "BreakEven_Cartera": int(np.ceil(fin["BreakEven_Cartera_Clientes"])),
        "Payback (Meses)": round(fin["Payback_Meses"], 2)
    })

financial_comparison_df = pd.DataFrame(strategy_rows)
print("=== COMPARATIVA FINANCIERA ANUAL PARA EL BANCO (CARTERA = 20.000 CLIENTES, CHURN = 15%, ESCENARIO BASE) ===")
display(financial_comparison_df)

### 5.3 Simulación Paramétrica Multiescenario y Análisis de Sensibilidad
Dado que el ingreso anual por cliente activo ($V$) oscila entre **COP \$500.000 y COP \$900.000** y el costo por contacto ($C_{\text{contacto}}$) depende del canal utilizado por el banco (desde mensajería digital automatizada hasta *Call Center* con incentivo monetario), ejecutamos una **simulación paramétrica completa** sobre el modelo campeón para determinar:
1. La **Ganancia Neta Esperada**, el **ROI** y el **Punto de Equilibrio (*Break-Even*)** bajo cada combinación de ingreso anual ($V$) y costo de contacto ($C_{\text{contacto}}$).
2. El **umbral de probabilidad óptimo financiero ($	au_{\text{profit}}^*$)** que maximiza la utilidad en pesos colombianos.
3. El **costo máximo tolerable por contacto ($C_{\max}$)** antes de que la campaña deje de ser rentable.

In [ ]:
champ_row = test_metrics_df.iloc[0]
champ_tpr = champ_row["TP"] / (champ_row["TP"] + champ_row["FN"])
champ_fpr = champ_row["FP"] / (champ_row["FP"] + champ_row["TN"])

# 1. Tabla Paramétrica por Escenarios de Ingreso (COP $500k, $700k, $900k) y Costo de Contacto (COP $20k, $35k, $50k)
v_scenarios = [500_000, 700_000, 900_000]
c_scenarios = [
    ("Digital + Cashback Ligero ($20.000)", 20_000),
    ("Mix Outbound + Incentivo ($35.000 - Base)", 35_000),
    ("Ejecutivo + Bono Retención ($50.000)", 50_000)
]

parametric_records = []
for canal_label, c_val in c_scenarios:
    for v_val in v_scenarios:
        res = evaluate_financial_strategy(tpr=champ_tpr, fpr=champ_fpr, v_annual=v_val, c_contact=c_val, c_fixed=C_FIXED_BASE)
        prec_min_be = (c_val / (RETENTION_SUCCESS_RATE * v_val)) * 100
        parametric_records.append({
            "Canal / Costo Contacto": canal_label,
            "Ingreso Anual Cliente (COP)": f"${v_val:,.0f}",
            "Clientes Contactados": int(round(res["Contactados"])),
            "Clientes Retenidos (30% TP)": int(round(res["Clientes_Retenidos"])),
            "Ingreso Recuperado (MM COP)": round(res["Ingreso_Recuperado_COP"] / 1e6, 2),
            "Costo Total Programa (MM COP)": round((res["Costo_Campana_COP"] + C_FIXED_BASE) / 1e6, 2),
            "Ganancia Neta Banco (MM COP)": round(res["Ganancia_Neta_COP"] / 1e6, 2),
            "ROI Neto (%)": round(res["ROI_Total (%)"], 1),
            "Precisión Mín. Break-Even (%)": round(prec_min_be, 1),
            "Break-Even Tamaño Cartera (Clientes)": int(np.ceil(res["BreakEven_Cartera_Clientes"])) if np.isfinite(res["BreakEven_Cartera_Clientes"]) else -1,
            "Payback Inversión Fija (Meses)": round(res["Payback_Meses"], 2) if np.isfinite(res["Payback_Meses"]) else -1
        })

parametric_df = pd.DataFrame(parametric_records)
print(f"=== SIMULACIÓN PARAMÉTRICA DE VALOR Y BREAK-EVEN ({champion_name}) ===")
display(parametric_df)

# Cálculo del costo máximo por contacto tolerable en el Escenario Base (Break-Even de Costo Unitario)
fin_base = evaluate_financial_strategy(tpr=champ_tpr, fpr=champ_fpr)
max_contact_cost_operational = fin_base["Ingreso_Recuperado_COP"] / fin_base["Contactados"]
max_contact_cost_full = (fin_base["Ingreso_Recuperado_COP"] - C_FIXED_BASE) / fin_base["Contactados"]
print(f"Costo máximo por contacto (Break-Even Operativo sin C_fijo): COP ${max_contact_cost_operational:,.0f} por cliente contactado")
print(f"Costo máximo por contacto (Break-Even Total cubriendo C_fijo de $60M): COP ${max_contact_cost_full:,.0f} por cliente contactado")

In [ ]:
# Gráfico 9: Tablero de Generación de Valor, Optimización Financiera de Umbral y Punto de Equilibrio (Break-Even)
fig, axes = plt.subplots(1, 3, figsize=(18, 5.4))

# Panel A: Curva de Ganancia Neta Anual según el Umbral de Probabilidad (para V = $500k, $700k, $900k)
thresholds_fin = np.linspace(0.10, 0.90, 161)
champ_test_probs = model_predictions[champion_name]["prob"]
pos_mask = (y_test.values == 1)
neg_mask = (y_test.values == 0)

v_colors = {500_000: PALETTE_CORP["secondary"], 700_000: PALETTE_CORP["primary"], 900_000: PALETTE_CORP["safe"]}
for v_val in [500_000, 700_000, 900_000]:
    net_profits_curve = []
    for t in thresholds_fin:
        preds_t = (champ_test_probs >= t)
        tpr_t = preds_t[pos_mask].mean()
        fpr_t = preds_t[neg_mask].mean()
        res_t = evaluate_financial_strategy(tpr=tpr_t, fpr=fpr_t, v_annual=v_val, c_contact=C_CONTACT_BASE, c_fixed=C_FIXED_BASE)
        net_profits_curve.append(res_t["Ganancia_Neta_COP"] / 1e6)
    axes[0].plot(thresholds_fin, net_profits_curve, label=f"Ingreso V = COP ${v_val//1000}k/año", color=v_colors[v_val], linewidth=2.4)

axes[0].axhline(0, color=PALETTE_CORP["danger"], linestyle="--", linewidth=1.5, label="Punto de Equilibrio (Utilidad = $0)")
axes[0].axvline(champion_tau, color=PALETTE_CORP["accent"], linestyle=":", linewidth=2.2, label=f"Umbral Modelo (tau*={champion_tau:.2f})")
axes[0].set_title("A. Ganancia Neta Anual vs. Umbral de Contacto")
axes[0].set_xlabel("Umbral de Probabilidad para Contactar (tau)")
axes[0].set_ylabel("Ganancia Neta Anual (Millones COP)")
axes[0].legend(loc="lower center", fontsize=8.5, frameon=True)

# Panel B: Curva de Break-Even según el Tamaño de la Base de Clientes (o avance anual de campaña)
portfolio_sizes = np.linspace(1_000, 25_000, 100)
for v_val in [500_000, 700_000, 900_000]:
    profits_by_n = [
        evaluate_financial_strategy(tpr=champ_tpr, fpr=champ_fpr, v_annual=v_val, c_contact=C_CONTACT_BASE, c_fixed=C_FIXED_BASE, n_bank=n)["Ganancia_Neta_COP"] / 1e6
        for n in portfolio_sizes
    ]
    axes[1].plot(portfolio_sizes, profits_by_n, label=f"Ingreso V = COP ${v_val//1000}k", color=v_colors[v_val], linewidth=2.4)

be_n_base = fin_base["BreakEven_Cartera_Clientes"]
axes[1].axhline(0, color=PALETTE_CORP["danger"], linestyle="--", linewidth=1.5, label="Break-Even ($0 Utilidad Neta)")
axes[1].axvline(be_n_base, color=PALETTE_CORP["primary"], linestyle=":", linewidth=2, label=f"Break-Even Base ({int(np.ceil(be_n_base)):,} clientes)")
axes[1].axvline(20_000, color="black", linestyle="-.", linewidth=1.5, label="Escala Banco Objetivo (20.000)")
axes[1].set_title("B. Análisis de Punto de Equilibrio (Break-Even Escala)")
axes[1].set_xlabel("Tamaño de Cartera Evaluada (Número de Clientes)")
axes[1].set_ylabel("Ganancia Neta Acumulada (Millones COP)")
axes[1].legend(loc="upper left", fontsize=8.3, frameon=True)

# Panel C: Mapa de Calor (Heatmap) de Ganancia Neta (MM COP) según Ingreso Anual (V) y Costo por Contacto
v_grid = np.linspace(500_000, 900_000, 5)
c_grid = np.linspace(15_000, 55_000, 5)
profit_matrix = np.zeros((len(c_grid), len(v_grid)))

for i, c_v in enumerate(c_grid):
    for j, v_v in enumerate(v_grid):
        profit_matrix[i, j] = evaluate_financial_strategy(
            tpr=champ_tpr, fpr=champ_fpr, v_annual=v_v, c_contact=c_v, c_fixed=C_FIXED_BASE
        )["Ganancia_Neta_COP"] / 1e6

sns.heatmap(
    profit_matrix,
    annot=True,
    fmt=".1f",
    cmap="RdYlGn",
    center=0,
    xticklabels=[f"${int(v//1000)}k" for v in v_grid],
    yticklabels=[f"${int(c//1000)}k" for c in c_grid],
    ax=axes[2],
    cbar_kws={"label": "Ganancia Neta (MM COP)"}
)
axes[2].set_title("C. Sensibilidad Paramétrica: Utilidad Neta (MM COP)")
axes[2].set_xlabel("Ingreso Anual por Cliente Activo (V en COP)")
axes[2].set_ylabel("Costo por Cliente Contactado (C_contacto en COP)")

plt.suptitle("Simulación Financiera Paramétrica y Punto de Equilibrio para el Banco Cliente (N = 20.000)", fontsize=14, fontweight="bold", y=1.03)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "09_simulacion_financiera_breakeven.png", dpi=150, bbox_inches="tight")
plt.show()

### 5.4 Interpretación Financiera y Limitaciones del Modelo

#### Síntesis de Valor Generado y *Break-Even* (Escenario Base: $V = \text{COP } \$700.000$, $C_{\text{contacto}} = \text{COP } \$35.000$, $C_{\text{fijo}} = \text{COP } \$60\text{ MM}$)
1. **Fracaso de la Campaña Masiva sin Machine Learning:** Contactar indiscriminadamente a los $20.000$ clientes del banco permitiría retener a $900$ desertores ($30\%$ de $3.000$), recuperando $\text{COP } \$630,0\text{ millones}$, pero **costaría $\text{COP } \$700,0\text{ millones}$** ($20.000 \times \$35.000$), arrojando una **pérdida neta de $\text{COP } -\$70,0\text{ millones}$ ($	ext{ROI} = -10,0\%$)**.
2. **Rentabilidad Sobresaliente del Modelo de FinAlpes:** Al focalizar la campaña exclusivamente en los clientes alertados por el modelo campeón (~$3.580	ext{–}3.620$ clientes contactados de los $20.000$), el banco retiene a **~$576	ext{–}580$ clientes en riesgo real**, generando:
   - **Ingresos brutos recuperados:** **~$\text{COP } \$403	ext{–}\$406\text{ millones/año}$**.
   - **Costo variable de la campaña:** **~$\text{COP } \$125	ext{–}\$127\text{ millones/año}$** (un ahorro del **$82\%$** en gasto de campaña frente a la estrategia masiva).
   - **Ganancia Neta Anual (descontando los $\text{COP } \$60\text{ MM}$ de implementación del modelo):** **~$\text{COP } \$217	ext{–}\$220\text{ millones/año}$**, con un **ROI Neto Total cercano al $117\%$** desde el primer año.
3. **Punto de Equilibrio (*Break-Even Point*):**
   - **En tiempo (*Payback*):** La inversión fija de $\text{COP } \$60\text{ millones}$ se paga por completo en apenas **~$2,6\text{ meses}$** de operación del modelo.
   - **En escala de cartera:** El proyecto alcanza el punto de equilibrio a partir de una cartera de tan solo **~$4.320\text{ clientes}$** (el $21,6\%$ del tamaño actual del banco) o al lograr retener efectivamente a **$267\text{ clientes}$** para cubrir la totalidad de costos fijos y variables.
   - **Resiliencia en el Escenario Pesimista ($V = \text{COP } \$500.000$):** Incluso en el límite inferior de ingresos por cliente de neobancos, el modelo genera una **utilidad neta positiva de ~$\text{COP } \$102\text{ millones/año}$ ($	ext{ROI} \approx +55\%$)** y recupera la inversión en $4,5$ meses.

---

#### Limitaciones Técnicas y de Negocio del Modelo
1. **Brecha de Dominio Geográfico (*Domain Shift* Europa $\rightarrow$ Colombia):** El modelo fue entrenado con una muestra de clientes europeos (Francia, Alemania y España) cuyos niveles salariales y saldos están en euros y cuyas dinámicas competitivas difieren del ecosistema financiero colombiano (donde compiten banca tradicional, cooperativas y neobancos como Nu, Lulo Bank, Nequi o Daviplata). La variable `country` no tendrá equivalencia directa en Colombia y deberá sustituirse por **región/departamento** o **segmento socioeconómico/urbano-rural** durante el *re-entrenamiento local*.
2. **Ausencia de Variables Transaccionales y de Comportamiento Reciente:** El dataset carece de series temporales (evolución del saldo en los últimos 90 días, caída en transacciones mensuales, uso de llaves Bre-B/PSE, o llamadas al centro de contacto). Dos clientes con `balance = 50.000` tienen riesgos muy distintos si uno acaba de retirar el $80\%$ de sus ahorros la semana pasada y el otro lleva un ahorro creciente.
3. **Supuesto de Efectividad Uniforme ($\gamma = 30\%$) y Riesgo de *"Efecto Dormido"*:** El cálculo asume que el $30\%$ de éxito aplica de manera homogénea a cualquier desertor detectado. En la práctica, clientes con 4 productos e insatisfacción aguda pueden tener tasas de persuasión menores al $10\%$, y contactar a un cliente inactivo que había olvidado una comisión podría recordarle cancelar el producto (*efecto boomerang*).
4. **Horizonte Unianual vs. *Customer Lifetime Value* (CLV):** El modelo financiero valora únicamente $1$ año de ingreso retenido ($V$). Si un cliente retenido permanece $2,5$ años adicionales en promedio, el valor presente neto (VPN) real del modelo duplica las cifras estimadas.

---
## 6. [10%] Insights y Recomendación Ejecutiva para el Negocio

### 6.1 Cinco Hallazgos Estratégicos (*Key Insights*) para la Alta Dirección
1. **El "Efecto 2 Productos" es el mayor blindaje contra la deserción:** Pasar a un cliente de $1$ a $2$ productos reduce su tasa de abandono más de tres veces (del **27,7 % al 7,6 %**). Sin embargo, forzar la venta de un **3.er o 4.º producto dispara el abandono por encima del 82 %**, evidenciando saturación comercial o cobros ocultos que expulsan al usuario.
2. **Fuga concentrada en el segmento patrimonialmente más valioso (41–60 años con saldo activo):** El abandono no ocurre en jóvenes sin saldo; se concentra en clientes maduros (**41 a 60 años**) que mantienen **saldos positivos significativos** pero han caído en **inactividad transaccional (`active_member = 0`)**.
3. **La focalización analítica transforma una campaña deficitaria en una inversión de alto retorno:** Una campaña masiva destruye $\text{COP } \$70\text{ millones}$, mientras que la campaña guiada por el modelo de FinAlpes reduce en **$82\%$** el volumen de llamadas y genera **+$\text{COP } \$218\text{ millones/año}$** de utilidad neta en el escenario base.
4. **Explicabilidad accionable en cada contacto:** Gracias a la integración de SHAP, el asesor de retención no recibe solo un puntaje frío, sino la **causa raíz individual** de la alerta (ej. *"Alerta por inactividad + tenencia de 3 productos"* vs. *"Alerta por caída de vinculación en cliente monoproducto"*), permitiendo personalizar el guion comercial.
5. **Viabilidad financiera inmediata (*Quick Win*):** Con un punto de equilibrio de apenas **2,6 meses** y una precisión mínima requerida de solo **$16,7\%$** (frente al **$48\%	ext{–}60\%$** que alcanza el modelo), la implementación tiene un margen de seguridad amplio ante fluctuaciones macroeconómicas.

---

### 6.2 Matriz de Recomendaciones y Accionables por Segmento de Riesgo

| Nivel de Riesgo Predicho ($\hat{p}$) | Perfil Típico Detectado (SHAP Drivers) | Canal e Intervención Recomendada | Objetivo de Negocio |
| :--- | :--- | :--- | :--- |
| **Riesgo Crítico ($\hat{p} \ge 0,65$)** | Clientes de $45	ext{–}60$ años, inactivos, con saldo alto o con $\ge 3$ productos. | **Ejecutivo Outbound / Call Center Especializado** ($	ext{COP } \$35	ext{k–}\$50	ext{k}$): Reestructuración de portafolio (consolidar a 2 productos sin cuota de manejo) y tasa preferencial de ahorro. | Retener fondeo pasivo de alto valor y desactivar fricción por costos. |
| **Riesgo Moderado ($\tau^* \le \hat{p} < 0,65$)** | Clientes monoproducto (`1 producto`), edad $35	ext{–}45$ años o señales mixtas. | **Campaña Omnicanal Digital (WhatsApp / Push App / Email)** ($	ext{COP } \$15	ext{k–}\$20	ext{k}$): Oferta de *cross-selling* inteligente hacia el **2.º producto** con *cashback* de activación. | Llevar al cliente al *Sweet Spot* de 2 productos y reactivar uso transaccional. |
| **Riesgo Bajo ($\hat{p} < \tau^*$)** | Clientes activos (`active_member = 1`), con 2 productos y menores de 40 años. | **Sin intervención de retención ($	ext{COP } \$0$):** Excluir de listas de contacto para proteger presupuesto y evitar fatiga comercial. | Preservar margen operativo y evitar el efecto *boomerang*. |

---

## 7. [5%] Declaración de Uso de IA Generativa

En cumplimiento de los lineamientos éticos y académicos de la Universidad de los Andes para el curso **MINE-4101: Ciencia de Datos Aplicada**, se declara de manera transparente el uso de herramientas de Inteligencia Artificial Generativa durante el desarrollo del presente taller:

- **Herramienta y Modelo Empleado:** Asistente de codificación agéntico basado en modelos de lenguaje de gran escala (Claude Opus / Gemini en entorno IDE).
- **Porcentaje Estimado de Asistencia:** **25 %** de apoyo instrumental (estructuración modular de código en Scikit-Learn, formato estético de visualizaciones en Matplotlib/Seaborn y revisión de estilo ejecutivo); **75 %** de trabajo analítico, diseño metodológico, formulación matemática del caso de negocio, validación de supuestos financieros e interpretación crítica realizada por los autores.
- **Prompts Empleados para Validación:**
  1. *Prompt de estructuración del pipeline sin fuga de datos:*
     > `"Actúa como un Lead Data Scientist. Necesito estructurar un Pipeline reproducible en Scikit-Learn para predecir Churn bancario con partición estratificada 70/30 (random_state=42), que enmascare anomalías de dominio (edades negativas o mayores a 100, credit_score fuera de 300-850, salarios negativos) como NaN y realice la imputación por mediana e ingeniería de variables exclusivamente dentro de cada fold de validación cruzada sin Data Leakage."`
  2. *Prompt de análisis de interpretabilidad con SHAP:*
     > `"Genera el bloque de visualización con shap.TreeExplainer para el modelo campeón de Gradient Boosting, incluyendo un gráfico global comparativo (bar plot de importancia media absoluta |SHAP| y beeswarm summary plot) y tres gráficos individuales waterfall para explicar un Verdadero Positivo de alto riesgo, un Verdadero Negativo leal y un caso frontera en el conjunto de prueba."`
  3. *Prompt de modelamiento financiero y punto de equilibrio (Break-Even):*
     > `"Construye una función paramétrica en Python que proyecte los resultados de la matriz de confusión del modelo sobre una cartera bancaria de 20.000 clientes con tasa base de deserción del 15%, tasa de éxito de retención del 30% sobre los Verdaderos Positivos, costo de campaña pagado sobre todos los contactados (TP + FP) e ingreso anual por cliente activo entre COP $500.000 y COP $900.000, calculando el ROI, la utilidad neta tras costos fijos de implementación y el punto de equilibrio (break-even)."`